<a href="https://colab.research.google.com/github/fataa34/applied-optimization-with-python/blob/unconstraint/Uncontraint_Non_Gradient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unconstraint Non-Gradient

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def fungsi_objektif(x):
  return 3+(x[0]-1.5*x[1])**2+(x[1]-2)**2

In [ ]:
def golden_section(fu_ob,a, b, epsilon, max_iter):
  tau = 0.38197
  results = []
  x1 = (1 - tau) * a + tau * b
  x2 = tau * a + (1 - tau) * b
  f1 = fu_ob(x1)
  f2 = fu_ob(x2)
  for i in range(max_iter):
    if abs(b - a) < epsilon:
      break
    elif f1 > f2:
      a = x1
      x1 = x2
      f1 = f2
      x2 = tau * a + (1 - tau) * b
      f2 = fu_ob(x2)
    else:
      b = x2
      x2 = x1
      f2 = f1
      x1 = (1 - tau) * a + tau * b
      f1 = fu_ob(x1)
  return (a + b) / 2

## Random Walk

In [ ]:
def random_walk(x0, max_iter, epsilon, a, b):
  results = []
  x_baru = x0
  for i in range(max_iter):
    s_i = np.random.randn(len(x_baru))
    s_i = s_i / np.linalg.norm(s_i)

    def g(alpha):
      return fungsi_objektif(x_baru + alpha * s_i)

    alpha = golden_section(g, a, b, epsilon, 100)
    x_baru = x_baru + alpha * s_i
    f_x = fungsi_objektif(x_baru)
    results.append([i, x_baru, f_x])
  return pd.DataFrame(results, columns=["Iterasi", "x", "fungsi x"])

In [ ]:
random_walk([0.5, 0.5], 20, 1e-2, -10, 10)

,Iterasi,x,fungsi x
0,0,"[0.39739639543304195, 0.5582652317336638]",5.272200
1,1,"[0.3565083233609708, 0.7279195596624438]",5.158959
2,2,"[0.7120226807832686, 0.6391361466972005]",4.912802
3,3,"[1.4113270182300217, 1.472670100319228]",3.914367
4,4,"[1.7447298109042904, 1.7474154785188194]",3.831864
5,5,"[2.4127256784982665, 2.145259435693397]",3.669389
6,6,"[2.134664206519825, 1.6675058813029504]",3.244944
7,7,"[2.562881591122595, 1.9685980478146776]",3.153098
8,8,"[2.6418290961518003, 1.8192665591615365]",3.040246
9,9,"[2.909658407480109, 1.9988756825130367]",3.007861


## Pattern Search

In [ ]:
def pattern_search(x0, eps1, eps2, a, b, epsilon, Nc):
    x0 = np.asarray(x0, dtype=float)
    n = x0.size

    Xc = x0.copy()
    f_c = fungsi_objektif(Xc)

    iter_history = []

    for cycle in range(1, Nc + 1):
        X = Xc.copy()
        f_before_cycle = f_c
        alphas = np.zeros(n)
        S_list = []

        # coordinate searches
        for i in range(n):
            S_i = np.zeros(n)
            S_i[i] = 1.0

            def g(alpha):
                return fungsi_objektif(X + alpha * S_i)

            alpha_i = golden_section(g, a, b, epsilon, 100)

            X_new = X + alpha_i * S_i
            f_after = fungsi_objektif(X_new)

            alphas[i] = alpha_i
            S_list.append(alpha_i * S_i)
            X = X_new

        X_n1 = X.copy()

        # pattern step
        S_j = np.sum(S_list, axis=0)
        if np.linalg.norm(S_j) < 1e-16:
            alpha_j = 0.0
            X_j = X_n1.copy()
            f_after_pattern = fungsi_objektif(X_j)
        else:
            def g_pattern(alpha):
                return fungsi_objektif(X_n1 + alpha * S_j)

            alpha_j = golden_section(g_pattern, a, b, epsilon, 100)
            X_j = X_n1 + alpha_j * S_j
            f_after_pattern = fungsi_objektif(X_j)

        iter_history.append({
            "Cycle": cycle,
            "x_before": X_n1.tolist(),
            "x_after": X_j.tolist(),
            "f_x_after": float(f_after_pattern)
        })

        delta_f = f_after_pattern - f_before_cycle
        delta_x = np.asarray(X_j) - np.asarray(Xc)
        delta_x_norm2 = float(np.dot(delta_x, delta_x))


        # update Xc, f_c for next cycle (or final state)
        Xc = np.asarray(X_j).copy()
        f_c = float(f_after_pattern)

        # stopping criteria
        if abs(delta_f) <= eps1 or delta_x_norm2 <= eps2:
            break
        # else continue for next cycle (for-loop will end naturally if cycle == Nc)

    iter_df = pd.DataFrame(iter_history, columns=[
    "Cycle", "x_before", "x_after", "f_x_after"
    ])

    return iter_df

In [ ]:
pattern_search([4, 3],1e-8, 1e-8, -10, 10, 1e-6, 50)

,Cycle,x_before,x_after,f_x_after
0,1,"[4.499995259527624, 2.692306120968711]","[4.386789458429815, 2.7619722456066484]",3.640055
1,2,"[4.1429575255445625, 2.527518895134584]","[3.8290102819093086, 2.225647100612929]",3.291546
2,3,"[3.338469589200891, 2.1562058568489926]","[3.1758393332868238, 2.1331838172665742]",3.018311
3,4,"[3.199776650679048, 2.0922145539526014]","[3.1958545509614025, 2.098927316850923]",3.012039
4,5,"[3.148386138476729, 2.068488474542502]","[3.037737947127752, 1.9975359679830935]",3.001723
5,6,"[2.996304426527038, 1.9982959128394462]","[2.9973833704268835, 1.9982761235998379]",3.000003
6,7,"[2.997413346849951, 1.99880195427276]","[2.9974144321885166, 1.9988209927119698]",3.000002
7,8,"[2.9982312583206387, 1.999184624424283]","[2.999995148622572, 1.999969866748936]",3.000000
8,9,"[2.9999547475595634, 1.999979271905451]","[2.9999623717063892, 1.9999774970438793]",3.000000


## Metode Powell

In [ ]:
def powell_method(x0, a, b, epsilon, epsilon1, epsilon2, max_cycle):

    n = len(x0)
    X1 = np.copy(x0)
    Xc = np.copy(x0)
    j = 1  # inisialisasi siklus Powell

    results = []
    # basis awal (arah satuan)
    S = np.eye(n)

    for j in range(max_cycle):
        X = np.copy(X1)
        f_prev = fungsi_objektif(X)

        # Step 2: Univariate search di sepanjang setiap arah Si
        for i in range(n):
            def phi(alpha):
                return fungsi_objektif(X + alpha * S[i])

            # line search (contoh sederhana: golden section)
            alpha_opt = golden_section(phi, a, b, epsilon,100)
            X = X + alpha_opt * S[i]

        # Langkah pola (pattern step)
        Sp = X - X1
        def phi_p(alpha):
            return fungsi_objektif(X + alpha * Sp)
        alpha_p = golden_section(phi_p, a, b, epsilon,100)
        Xp = X + alpha_p * Sp

        f_curr = fungsi_objektif(Xp)
        delta_f = abs(f_curr - f_prev)
        delta_x = np.linalg.norm(Xp - X1)

        results.append([j, X1, f_prev, Xp,f_curr])
        # kondisi berhenti
        if delta_f <= epsilon1 or delta_x <= epsilon2:
            print(f"Konvergen setelah {j+1} siklus.")
            return pd.DataFrame(results, columns=["j","X1","f_sebelumnya", "Xp", "f_hasil"])

        # update arah terakhir
        S[:-1] = S[1:]
        S[-1] = Sp

        X1 = np.copy(Xp)

    print("Mencapai jumlah siklus maksimum.")
    return pd.DataFrame(results, columns=["j","X1","f_sebelumnya", "Xp", "f_hasil"])

In [ ]:
  powell_method([0.5,0.5], -10, 10, 1e-6, 1e-8, 1e-8, 50)

Konvergen setelah 4 siklus.


,j,X1,f_sebelumnya,Xp,f_hasil
0,0,"[0.5, 0.5]",5.312500,"[0.8558978502170634, 1.157024068462622]",4.484372
1,1,"[0.8558978502170634, 1.157024068462622]",4.484372,"[1.9658923063436071, 1.5934604561182812]",3.345304
2,2,"[1.9658923063436071, 1.5934604561182812]",3.345304,"[3.0000194447084834, 2.000036008824521]",3.000000
3,3,"[3.0000194447084834, 2.000036008824521]",3.000000,"[3.000019559577431, 2.000036053986134]",3.000000
